# Logistic Regression (Binomial)

This notebook implements and demonstrates Logistic Regression (Binomial) based on a given set of training data samples.

## Importing Required Libraries

Let's import the necessary libraries for implementing logistic regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, roc_auc_score

# For reproducibility
np.random.seed(42)

## Loading the Dataset

We'll create a sample dataset for demonstration purposes. In a real scenario, you would load your data from a CSV file.

In [ ]:
# Create a sample dataset (diabetes prediction data)
# Features: Age, BMI, Glucose Level
# Target: Diabetes (1 = Yes, 0 = No)

# Sample data creation
np.random.seed(42)
n_samples = 100

# Create features with some correlation to the target
age = np.random.normal(45, 15, n_samples)
bmi = np.random.normal(26, 5, n_samples)
glucose = np.random.normal(100, 25, n_samples)

# Combine features with weights to create a probability
p = 1 / (1 + np.exp(-(0.03 * age + 0.07 * bmi + 0.02 * glucose - 7)))
diabetes = np.random.binomial(1, p)

# Create DataFrame
data = {
    'Age': age,
    'BMI': bmi,
    'Glucose': glucose,
    'Diabetes': diabetes
}

df = pd.DataFrame(data)

# Save to CSV for future use
df.to_csv('diabetes_data.csv', index=False)
print("Sample data saved to 'diabetes_data.csv'")

# Display the first few rows of the dataset
df.head()

In [ ]:
# Alternative: Load data from CSV file
# Uncomment the line below to load your own CSV file
# df = pd.read_csv('diabetes_data.csv')
# df.head()

## Data Exploration and Preprocessing

Let's explore the data and prepare it for logistic regression.

In [1]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())

# Look at basic statistics
print("\nBasic statistics:")
print(df.describe())

# Class distribution
print("\nClass distribution:")
print(df['Diabetes'].value_counts())
print(f"Percentage of diabetic cases: {df['Diabetes'].mean() * 100:.2f}%")

Missing values in each column:


NameError: name 'df' is not defined

In [ ]:
# Visualize the data
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
sns.boxplot(x='Diabetes', y='Age', data=df)
plt.title('Age vs Diabetes')

plt.subplot(1, 3, 2)
sns.boxplot(x='Diabetes', y='BMI', data=df)
plt.title('BMI vs Diabetes')

plt.subplot(1, 3, 3)
sns.boxplot(x='Diabetes', y='Glucose', data=df)
plt.title('Glucose vs Diabetes')

plt.tight_layout()
plt.show()

## Data Preparation for Modeling

Split the data into features (X) and target variable (y), and then into training and testing sets.

In [ ]:
# Separate features and target
X = df.drop('Diabetes', axis=1)
y = df['Diabetes']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

In [ ]:
# Standardize the features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to DataFrame to keep feature names (for interpretation)
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Display scaled data
X_train_scaled_df.head()

## Building and Training the Logistic Regression Model

Now we'll implement the logistic regression model using scikit-learn.

In [ ]:
# Create a logistic regression model
model = LogisticRegression(random_state=42)

# Train the model on the training data
model.fit(X_train_scaled, y_train)

# Make predictions on the test data
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]  # Probability estimates for the positive class

## Model Evaluation

Let's evaluate the performance of our logistic regression model.

In [ ]:
# Model accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

# Classification report
class_report = classification_report(y_test, y_pred)
print("\nClassification Report:")
print(class_report)

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='best')
plt.show()

## Model Interpretation

Interpreting the coefficients of the logistic regression model.

In [ ]:
# Get model coefficients
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
})

# Sort coefficients by absolute value for better visualization
coefficients = coefficients.reindex(coefficients['Coefficient'].abs().sort_values(ascending=False).index)

print("Model Coefficients:")
print(coefficients)

# Visualize coefficients
plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=coefficients)
plt.title('Logistic Regression Coefficients')
plt.axvline(x=0, color='k', linestyle='--')
plt.show()

## Making Predictions on New Data

Let's use our trained model to predict diabetes for a new individual.

In [ ]:
# Function to predict diabetes for a new individual
def predict_diabetes(age, bmi, glucose):
    # Create a DataFrame for the new data
    new_data = pd.DataFrame({
        'Age': [age],
        'BMI': [bmi],
        'Glucose': [glucose]
    })
    
    # Scale the new data using the same scaler
    new_data_scaled = scaler.transform(new_data)
    
    # Make prediction
    prediction = model.predict(new_data_scaled)
    probability = model.predict_proba(new_data_scaled)[0, 1]  # Probability of being diabetic
    
    return prediction[0], probability

# Example: Predict diabetes for a new individual
age = 55
bmi = 32
glucose = 140

prediction, probability = predict_diabetes(age, bmi, glucose)

print(f"New individual - Age: {age}, BMI: {bmi}, Glucose: {glucose}")
print(f"Prediction: {'Diabetic' if prediction == 1 else 'Not Diabetic'}")
print(f"Probability of being diabetic: {probability:.4f}")

## Conclusion

In this notebook, we have implemented a logistic regression model for binary classification to predict diabetes based on age, BMI, and glucose level. We demonstrated:
1. Data preparation and exploration
2. Model training and evaluation
3. Interpretation of model coefficients
4. Making predictions on new data

Logistic regression is a simple yet powerful algorithm for binary classification problems. It provides easily interpretable results through its coefficients, making it a popular choice for many applications in medicine, finance, and other fields.